# multiply-back — worked example 2: multiply_back for scalar constant times a matrix

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `multiply-back`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When one operand of `x * y` is a Python scalar (float or int), it participates in broadcasting as if it were a tensor of shape `()`. The backward for the tensor operand is `grad_out * scalar` — straightforward. The backward for the scalar operand conceptually does not exist (scalars have no gradient), but the back function must still handle the case gracefully without crashing.

## Worked solution

**Step 1 — identify which operand is the scalar.** A Python `float` or `int` is not a Tensor. Check with `isinstance(operand, Tensor)`.

**Step 2 — compute the tensor-side gradient.** For `out = scalar * matrix`, `dL/d_matrix = grad_out * scalar`. Since `scalar` is a Python float, multiplication `grad_out * scalar` works directly via Python's numeric broadcasting — no coercion needed for the computation itself.

**Step 3 — wrap in `unbroadcast` for correctness.** Even though scalar multiplication doesn't change shape, calling `unbroadcast(grad_out * scalar, matrix)` is the right habit because `unbroadcast` is a no-op when shapes already match.

**Step 4 — handle the scalar-side back function.** `multiply_back1` would be called for the scalar operand. Since scalars have no shape, return a zero-dimensional tensor rather than crashing.

In [ ]:
import torch
from torch import Tensor

torch.manual_seed(0)

def unbroadcast(grad: Tensor, original) -> Tensor:
    if not isinstance(original, Tensor):
        return grad.sum()  # scalar operand: collapse everything
    while grad.ndim > original.ndim:
        grad = grad.sum(dim=0)
    for i, (gs, os) in enumerate(zip(grad.shape, original.shape)):
        if os == 1 and gs != 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad

def multiply_back0(grad_out: Tensor, out: Tensor, x, y) -> Tensor:
    """dL/dx — x may be a tensor; y may be a Python float."""
    if not isinstance(y, Tensor):
        y = torch.tensor(y, dtype=grad_out.dtype)
    return unbroadcast(grad_out * y, x)

def multiply_back1(grad_out: Tensor, out: Tensor, x, y) -> Tensor:
    """dL/dy — y may be a Python float (returns 0-d tensor)."""
    if not isinstance(x, Tensor):
        x = torch.tensor(x, dtype=grad_out.dtype)
    if not isinstance(y, Tensor):
        # float side: collapse fully to 0-d
        return (grad_out * x).sum()
    return unbroadcast(grad_out * x, y)

# Exercise: out = 3.7 * M, then out = M * 3.7
torch.manual_seed(5)
M = torch.randn(3, 4)
grad_out = torch.randn(3, 4)

scale = 3.7
out = scale * M  # (3,4)

gM_0 = multiply_back0(grad_out, out, scale, M)  # dL/d_scale operand
gM_1 = multiply_back1(grad_out, out, scale, M)  # dL/dM

# Compare gM_1 against autograd
M2 = M.clone().requires_grad_(True)
(scale * M2).backward(grad_out)
print(f"Our gM shape: {gM_1.shape}, autograd shape: {M2.grad.shape}")
assert torch.allclose(gM_1, M2.grad, atol=1e-6)
print("Scalar multiply_back matches autograd!")